# Minimal GR00T-N1.7 fine-tune on `stack_without_ft_tact_v4`

Standalone notebook, no `launch_finetune.py` subprocess, no sbatch wrapper.
Every hyperparameter, path and split is read from `configs/finetune_minimal.yaml`
through `training_cfgs.Config` -- none is hardcoded in the notebook. Data loading goes through `robotdataset.LeRobotVideoDataset`
(the package's own LeRobot-v2.1-with-external-video loader); the model and its
processor are GR00T's own tested classes, used directly rather than through the
`Gr00tN1d7Pipeline`/`DatasetFactory`/HF-`Trainer` machinery.

Grounded facts from earlier characterization (`outputs/data_stats.csv`,
`gr00t_contract.md`):
- 51 episodes, 12,244 frames, 15 fps, single task "pick the lego block."
- EEF action (dims 0:6) is **RELATIVE** (state delta); gripper (dim 6) is
  **ABSOLUTE** continuous.
- No precomputed statistics are needed (`meta/stats.json` and `meta/relative_stats.json` are not read):
  the state/action and relative-action statistics are computed by `robotdataset` over the training
  episodes (section "Normalization statistics from robotdataset"; `EmbodimentTag.NEW_EMBODIMENT`,
  `ActionFormat.XYZ_ROTVEC` for the 6-dim pose).

Run this inside `$HPCWORK/gr00t-ft/venvs/gr00t` (`module load FFmpeg/7.1.2` first).

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
import wandb
from training_cfgs import Config

REPO_ROOT = Path.home() / "finetune-groot"
sys.path.insert(0, str(REPO_ROOT / "vendor" / "Isaac-GR00T"))
sys.path.insert(0, str(REPO_ROOT))  # for finetune_groot package (pyproject.toml at repo root)

# Single source of truth for hyperparameters, paths and the train/val split.
cfg = Config.from_file(REPO_ROOT / "configs" / "finetune_minimal.yaml")
print(cfg)

DATASET_PATH = Path(cfg.training.dataset_path)
CHECKPOINT_DIR = Path(cfg.training.output_dir) / cfg.training.experiment_name
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

Config
  model:
    base_model_path: 'nvidia/GR00T-N1.7-3B' (str)
    embodiment_tag: 'NEW_EMBODIMENT' (str)
    modality_config_path: '/home/yn030245/finetune-groot/embodiment_config.py' (str)
    tune_llm: False (bool)
    tune_visual: False (bool)
    tune_projector: True (bool)
    tune_diffusion_model: True (bool)
    load_bf16: True (bool)
  data:
    val_episodes: [46, 47, 48, 49, 50] (list)
  training:
    dataset_path: '/hpcwork/yn030245/gr00t-ft/data/stack_without_ft_tact_v4' (str)
    output_dir: '/hpcwork/yn030245/gr00t-ft/checkpoints' (str)
    experiment_name: 'notebook_run' (str)
    global_batch_size: 8 (int)
    learning_rate: 0.0001 (float)
    max_steps: 200 (int)
    warmup_ratio: 0.05 (float)
    max_grad_norm: 1.0 (float)
    save_steps: 100 (int)
    logging_steps: 10 (int)
    use_wandb: True (bool)
    wandb_project: 'finetune-gr00t-stack-tact' (str)
device: cuda


/work/yn030245/venv/finetune-groot/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from finetune_groot.embodiment_config import MODALITY_CONFIGS
from gr00t.data.embodiment_tags import EmbodimentTag

EMBODIMENT_TAG = EmbodimentTag.resolve(cfg.model.embodiment_tag)
modality_configs = MODALITY_CONFIGS[EMBODIMENT_TAG.value]
modality_configs

{'video': ModalityConfig(delta_indices=[0], modality_keys=['primary', 'wrist'], sin_cos_embedding_keys=None, mean_std_embedding_keys=None, action_configs=None),
 'state': ModalityConfig(delta_indices=[0], modality_keys=['cartesian', 'gripper', 'joints', 'target'], sin_cos_embedding_keys=None, mean_std_embedding_keys=None, action_configs=None),
 'action': ModalityConfig(delta_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], modality_keys=['eef', 'gripper'], sin_cos_embedding_keys=None, mean_std_embedding_keys=None, action_configs=[ActionConfig(rep=<ActionRepresentation.RELATIVE: 'relative'>, type=<ActionType.EEF: 'eef'>, format=<ActionFormat.XYZ_ROTVEC: 'xyz+rotvec'>, state_key='cartesian'), ActionConfig(rep=<ActionRepresentation.ABSOLUTE: 'absolute'>, type=<ActionType.NON_EEF: 'non_eef'>, format=<ActionFormat.DEFAULT: 'default'>, state_key='gripper')]),
 'language': ModalityConfig(delta_indices=[0], modality_keys=['annotation.human.task_description'], sin_cos_embedding_k

## Build the model

`Gr00tN1d7Config` defaults + the tuning flags and precision from the `model:`
group of the YAML (calibration-validated: projector + diffusion head trainable,
backbone frozen, bf16).


In [13]:
from gr00t.configs.model.gr00t_n1d7 import Gr00tN1d7Config
from gr00t.model.gr00t_n1d7.gr00t_n1d7 import Gr00tN1d7
from gr00t.model.gr00t_n1d7.processing_gr00t_n1d7 import Gr00tN1d7Processor

model_config = Gr00tN1d7Config(
    tune_llm=cfg.model.tune_llm,
    tune_visual=cfg.model.tune_visual,
    tune_projector=cfg.model.tune_projector,
    tune_diffusion_model=cfg.model.tune_diffusion_model,
)

model, loading_info = Gr00tN1d7.from_pretrained(
    cfg.model.base_model_path,
    tune_llm=model_config.tune_llm,
    tune_visual=model_config.tune_visual,
    tune_projector=model_config.tune_projector,
    tune_diffusion_model=model_config.tune_diffusion_model,
    tune_vlln=model_config.tune_vlln,
    state_dropout_prob=model_config.state_dropout_prob,
    backbone_trainable_params_fp32=model_config.backbone_trainable_params_fp32,
    load_bf16=cfg.model.load_bf16,
    output_loading_info=True,
    trust_remote_code=True,
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")


/work/yn030245/venv/finetune-groot/lib64/python3.12/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
`torch_dtype` is deprecated! Use `dtype` instead!
/home/yn030245/finetune-groot/vendor/Isaac-GR00T/gr00t/model/modules/dit.py:255: FutureWarning: Accessing config attribute `compute_dtype` directly via 'AlternateVLDiT' object attribute is deprecated. Please access 'compute_dtype' over 'AlternateVLDiT's config object instead, e.g. 'unet.config.compute_dtype'.
  embedding_dim=self.inner_dim, compute_dtype=self.compute_dtype
/home/yn030245/finetune-groot/vendor/Isaac-GR00T/gr00t/model/modules/dit.py:286: FutureWarning: Accessing config attribute `output_dim` directly via 'AlternateVLDiT' object attribute is deprecated. Please access 'output_dim' over

Total number of DiT parameters:  1091722240
Total number of SelfAttentionTransformer parameters:  201433088


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.66s/it]


trainable: 1,620,515,968 / 3,144,016,000 (51.5%)


## Build the processor

The processor is created without statistics. They are set after the datasets exist (see
"Normalization statistics from robotdataset" below), because robotdataset computes them over
the episodes the training dataset loaded. `processor.set_statistics` expects
`{modality: {joint_key: {stat: values}}}` plus a top-level `relative_action` entry.

In [ ]:
import json

processor = Gr00tN1d7Processor(
    modality_configs={EMBODIMENT_TAG.value: modality_configs},  # parse_modality_configs expects {tag: {modality: cfg}}
    use_percentiles=model_config.use_percentiles,
    statistics=None,
    embodiment_id_mapping=None,
    image_crop_size=model_config.image_crop_size,
    image_target_size=model_config.image_target_size,
    random_rotation_angle=model_config.random_rotation_angle,
    color_jitter_params=model_config.color_jitter_params,
    model_name=model_config.model_name,
    model_type=model_config.backbone_model_type,
    formalize_language=model_config.formalize_language,
    max_state_dim=model_config.max_state_dim,
    max_action_dim=model_config.max_action_dim,
    apply_sincos_state_encoding=model_config.apply_sincos_state_encoding,
    max_action_horizon=model_config.action_horizon,
    use_albumentations=model_config.use_albumentations_transforms,
    extra_augmentation_config=model_config.extra_augmentation_config,
    shortest_image_edge=model_config.shortest_image_edge,
    crop_fraction=model_config.crop_fraction,
    letter_box_transform=model_config.letter_box_transform,
    use_relative_action=model_config.use_relative_action,
    exclude_state=model_config.exclude_state,
    state_dropout_prob=model_config.state_dropout_prob,
    use_mean_std=model_config.use_mean_std,
)

## Load data via `robotdataset.LeRobotVideoDataset`

Every episode not in `data.val_episodes` trains (fixed split recorded in the YAML,
not randomized per run). Episode count and fps come from the dataset's
`meta/info.json`; the action horizon is the length of the registered
`action.delta_indices`, so `delta_timestamps` mirrors `embodiment_config.py`:
video/state anchor-only, action a multi-step window.


In [ ]:
import json

from robotdataset import LeRobotVideoDataset

info = json.loads((DATASET_PATH / "meta" / "info.json").read_text())
FPS = float(info["fps"])
ALL_EPISODES = list(range(info["total_episodes"]))
VAL_EPISODES = list(cfg.data.val_episodes)
TRAIN_EPISODES = [e for e in ALL_EPISODES if e not in VAL_EPISODES]

ACTION_HORIZON = len(modality_configs["action"].delta_indices)  # matches embodiment_config.py
BATCH_SIZE = cfg.training.global_batch_size

action_dt = [i / FPS for i in range(ACTION_HORIZON)]

train_ds = LeRobotVideoDataset(
    local_path=str(DATASET_PATH),
    episodes=TRAIN_EPISODES,
    batch_size=BATCH_SIZE,
    delta_timestamps={"action": action_dt},
    control_frequency=FPS,
)
print("train episodes:", train_ds.num_episodes, "total steps:", len(train_ds))

val_ds = LeRobotVideoDataset(
    local_path=str(DATASET_PATH),
    episodes=VAL_EPISODES,
    batch_size=BATCH_SIZE,
    delta_timestamps={"action": action_dt},
    control_frequency=FPS,
)
print("val episodes:", val_ds.num_episodes, "total steps:", len(val_ds))

Assembling training buffer: 100%|██████████| 46/46 [01:13<00:00,  1.59s/ep]


train episodes: 46 total steps: 11034


Assembling training buffer: 100%|██████████| 5/5 [00:04<00:00,  1.24ep/s]

val episodes: 5 total steps: 1210


: 

## Normalization statistics from robotdataset

Every statistic is computed by `robotdataset` over exactly the training episodes (everything except
`data.val_episodes`, so the held-out episodes do not leak into the normalization); nothing is read from
the dataset's `meta/stats.json` or `meta/relative_stats.json`.

- `train_ds.generate_stats(exact=True)`: state and action statistics; `to_gr00t_statistics` slices the
  joint groups with `meta/modality.json`.
- `train_ds.generate_relative_action_stats(...)`: statistics of the relative action chunks (for the
  `ActionRepresentation.RELATIVE` keys of the embodiment config, here `eef` relative to the state
  `cartesian`), per horizon step and dimension.

Exact quantiles matter: `use_percentiles` defaults to `True`, so the processor normalizes with the
1st/99th percentiles.

In [ ]:
import json
from pathlib import Path

from gr00t.data.types import ActionFormat, ActionRepresentation, ActionType
from robotdataset.gr00t import check_against, to_gr00t_statistics

modality_meta = json.loads((Path(DATASET_PATH) / "meta" / "modality.json").read_text())

# relative-action groups, read from the embodiment config
action_cfg = modality_configs["action"]
horizon = len(action_cfg.delta_indices)
assert list(action_cfg.delta_indices) == list(range(horizon))
groups, state_keys, representation = {}, {}, {}
for key, cfg_ in zip(action_cfg.modality_keys, action_cfg.action_configs):
    if cfg_.rep != ActionRepresentation.RELATIVE:
        continue
    span = modality_meta["action"][key]
    groups[key] = (span["start"], span["end"])
    state_keys[key] = "observation/state/" + (cfg_.state_key or key)
    if cfg_.type == ActionType.EEF:
        assert cfg_.format == ActionFormat.XYZ_ROTVEC, cfg_.format
        representation[key] = "xyz_rotvec"
    else:
        representation[key] = "joint"

statistics = to_gr00t_statistics(
    train_ds.generate_stats(exact=True),
    embodiment_tag=EMBODIMENT_TAG.value,
    modality_meta=modality_meta,
    relative_action=train_ds.generate_relative_action_stats(
        groups, state_keys, horizon=horizon, representation=representation, exact=True
    ),
)
check_against(statistics, modality_configs)
processor.set_statistics(statistics, override=True)
print(f"statistics from robotdataset over {train_ds.num_episodes} train episodes; processor has:",
      list(processor.statistics.keys()), "| relative groups:", list(groups))

## Adapter: robotdataset batch -> `VLAStepData` -> processor -> model inputs

`robotdataset` returns the raw 7-dim `action` column as one block (matching
the dataset's own parquet schema); it is sliced here into `eef` (dims 0:6)
and `gripper` (dim 6:7) to match `embodiment_config.py`'s two `ActionConfig`
entries. Video/state default to anchor-only (`T=1`), action carries the
16-step horizon.


In [ ]:
from finetune_groot.data_utils import batch_to_vla_steps, steps_to_model_batch

# Smoke-test the adapter on one batch before training.
_sample = train_ds.sample()
_steps = batch_to_vla_steps(_sample, embodiment_tag=EMBODIMENT_TAG)
_model_batch = steps_to_model_batch(_steps, processor)
print("model batch keys:", list(_model_batch.keys()) if hasattr(_model_batch, "keys") else type(_model_batch))


## Training loop

Minimal: AdamW over trainable params only, cosine schedule, wandb logging,
periodic checkpoint. No HF `Trainer` -- every step is explicit.


In [ ]:
MAX_STEPS = cfg.training.max_steps
WARMUP_STEPS = max(1, int(cfg.training.warmup_ratio * MAX_STEPS))

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg.training.learning_rate)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min(1.0, (step + 1) / WARMUP_STEPS)
    * max(0.0, (MAX_STEPS - step) / max(1, MAX_STEPS - WARMUP_STEPS)),
)

wandb.init(
    project=cfg.training.wandb_project,
    name=cfg.training.experiment_name,
    mode="online" if cfg.training.use_wandb else "disabled",
    config={**cfg.to_dict(), "action_horizon": ACTION_HORIZON, "trainable_params": trainable_params},
)

model.train()
processor.train()
losses = []
for step in range(MAX_STEPS):
    batch = train_ds.sample()
    steps = batch_to_vla_steps(batch, embodiment_tag=EMBODIMENT_TAG)
    model_batch = steps_to_model_batch(steps, processor)
    model_batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in model_batch.items()}

    # bf16 backbone + fp32 layer norms: run the forward under autocast (what HF Trainer does with bf16=True)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
        out = model(**model_batch)
    loss = out.loss if hasattr(out, "loss") else out["loss"]

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable, cfg.training.max_grad_norm)
    optimizer.step()
    scheduler.step()

    losses.append(loss.item())
    if step % cfg.training.logging_steps == 0 or step == MAX_STEPS - 1:
        wandb.log({"train/loss": loss.item(), "train/lr": scheduler.get_last_lr()[0]}, step=step)
        print(f"step {step:4d}  loss {loss.item():.4f}")
    if (step + 1) % cfg.training.save_steps == 0:
        ckpt_dir = CHECKPOINT_DIR / f"step_{step+1}"
        model.save_pretrained(ckpt_dir)
        processor.save_pretrained(ckpt_dir)
        print("saved checkpoint:", ckpt_dir)

wandb.finish()


## Loss curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses)
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.set_title("Minimal notebook training loop")
fig.tight_layout()
fig.savefig("notebook_run_loss.png", dpi=150)
plt.show()
